# Waste classifier — MobileNetV2 transfer learning

Fine-tunes a pretrained MobileNetV2 backbone on a folder dataset (TrashNet-style class directories). Does **not** train a CNN from scratch.

1. Download [TrashNet `dataset-resized.zip`](https://github.com/garythung/trashnet/blob/master/data/dataset-resized.zip) into `ml/dataset/`.
2. Unzip so you have `ml/dataset/dataset-resized/{cardboard,glass,metal,paper,plastic,trash}/`.
3. Run all cells. The exported model is picked up automatically by `backend/models/classifier.py`.

In [ ]:
from pathlib import Path
import json
import tensorflow as tf

DATA = Path("dataset/dataset-resized")
OUT = Path("saved_model")
OUT.mkdir(exist_ok=True)
IMG = 224
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG, IMG), batch_size=BATCH,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG, IMG), batch_size=BATCH,
)
class_names = list(train_ds.class_names)
print(class_names)
(OUT / "labels.json").write_text(json.dumps(class_names), encoding="utf-8")

autotune = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(autotune)
val_ds = val_ds.prefetch(autotune)

In [ ]:
n = len(class_names)
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet"
)
base.trainable = False
inputs = tf.keras.Input(shape=(IMG, IMG, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(n, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(train_ds, validation_data=val_ds, epochs=5)
model.save(OUT / "waste_classifier.keras")
print("saved", OUT / "waste_classifier.keras")